In [57]:
import pandas as pd
import numpy as np

In [58]:
df = pd.read_csv('../data/raw/results.csv')
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [59]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 49477 entries, 0 to 49476
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        49477 non-null  str    
 1   home_team   49477 non-null  str    
 2   away_team   49477 non-null  str    
 3   home_score  49433 non-null  float64
 4   away_score  49433 non-null  float64
 5   tournament  49477 non-null  str    
 6   city        49477 non-null  str    
 7   country     49477 non-null  str    
 8   neutral     49477 non-null  bool   
dtypes: bool(1), float64(2), str(6)
memory usage: 5.9 MB


In [60]:
df.isna().sum()

date           0
home_team      0
away_team      0
home_score    44
away_score    44
tournament     0
city           0
country        0
neutral        0
dtype: int64

In [61]:
df[df['home_score'].isna()]

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49433,2026-06-19,Scotland,Morocco,NaN,NaN,FIFA World Cup,Foxborough,United States,True
49434,2026-06-19,Brazil,Haiti,NaN,NaN,FIFA World Cup,Philadelphia,United States,True
49435,2026-06-19,United States,Australia,NaN,NaN,FIFA World Cup,Seattle,United States,False
49436,2026-06-19,Turkey,Paraguay,NaN,NaN,FIFA World Cup,Santa Clara,United States,True
49437,2026-06-20,Germany,Ivory Coast,NaN,NaN,FIFA World Cup,Toronto,Canada,True
49438,2026-06-20,Ecuador,Curaçao,NaN,NaN,FIFA World Cup,Kansas City,United States,True
49439,2026-06-20,Netherlands,Sweden,NaN,NaN,FIFA World Cup,Houston,United States,True
49440,2026-06-20,Tunisia,Japan,NaN,NaN,FIFA World Cup,Guadalupe,Mexico,True
49441,2026-06-21,Belgium,Iran,NaN,NaN,FIFA World Cup,Inglewood,United States,True
49442,2026-06-21,New Zealand,Egypt,NaN,NaN,FIFA World Cup,Vancouver,Canada,True


In [62]:
df.duplicated().sum()

np.int64(0)

In [63]:
df = df.dropna(subset=['home_score', 'away_score']).copy()

In [64]:
df['home_score'] = df['home_score'].astype(int)
df['away_score'] = df['away_score'].astype(int)

In [65]:
df["neutral"] = (df["neutral"].astype(str).str.upper()
                 .map({"TRUE": True, "FALSE": False}).fillna(False).astype(bool))

In [66]:
df["date"] = pd.to_datetime(df["date"])

In [67]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 49433 entries, 0 to 49432
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        49433 non-null  datetime64[us]
 1   home_team   49433 non-null  str           
 2   away_team   49433 non-null  str           
 3   home_score  49433 non-null  int64         
 4   away_score  49433 non-null  int64         
 5   tournament  49433 non-null  str           
 6   city        49433 non-null  str           
 7   country     49433 non-null  str           
 8   neutral     49433 non-null  bool          
dtypes: bool(1), datetime64[us](1), int64(2), str(5)
memory usage: 5.4 MB


In [68]:
df = df[df['date'] >= '2006-01-01'].copy()
df.shape

(19701, 9)

In [69]:
min_matches = 30

match_counts = pd.concat([df["home_team"], df["away_team"]]).value_counts()
keep_teams = set(match_counts[match_counts >= min_matches].index)
df = df[df["home_team"].isin(keep_teams) & df["away_team"].isin(keep_teams)].reset_index(drop=True)

# Dixon-Coles assumptions

In [70]:
nn = df[~df["neutral"]]
print("Avg home goals:", round(nn["home_score"].mean(), 3))
print("Avg away goals:", round(nn["away_score"].mean(), 3))

Avg home goals: 1.647
Avg away goals: 1.014


In [71]:
print("Corr(home, away goals):", round(df["home_score"].corr(df["away_score"]), 3))
pd.crosstab(df["home_score"].clip(upper=4),
            df["away_score"].clip(upper=4),
            normalize=True).round(3)

Corr(home, away goals): -0.182


away_score,0,1,2,3,4
home_score,,,,,
0,0.090,0.079,0.050,0.025,0.023
1,0.115,0.104,0.052,0.020,0.014
2,0.083,0.077,0.037,0.015,0.006
3,0.049,0.035,0.017,0.004,0.002
4,0.059,0.030,0.010,0.003,0.001


In [72]:
print("Neutral share:", round(df["neutral"].mean(), 3))
neu = df[df["neutral"]]
print("Neutral home-win %:", round((neu["home_score"] > neu["away_score"]).mean(), 3))
print("Neutral away-win %:", round((neu["home_score"] < neu["away_score"]).mean(), 3))

Neutral share: 0.285
Neutral home-win %: 0.405
Neutral away-win %: 0.344


In [73]:
df["tournament"].value_counts().head(10)

tournament
Friendly                                6343
FIFA World Cup qualification            4218
UEFA Euro qualification                 1322
African Cup of Nations qualification    1182
UEFA Nations League                      658
African Cup of Nations                   429
CONCACAF Nations League                  422
AFC Asian Cup qualification              368
FIFA World Cup                           348
Gold Cup                                 275
Name: count, dtype: int64

In [74]:
df.to_parquet("../data/processed/results_clean.parquet", index=False)